In [ ]:
# Alpha Connectome, sweep 7: training dynamics (step size 256/64/32 sequences x backprop length 500..20,000)
# One-cell launcher: takes the pipeline code from connectome_sweep6.ipynb (same Drive folder) and adds
# per-model mini-batches to trainer.py. 9 plain mask-weighted GRUs, 4 passes, judged on validation half B
# (reference: sweep 5 plain_m50 = 0.6755). Keep this tab open while it runs. Full version: colab/connectome_sweep7.ipynb.
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/wunder-connectome/runs'
os.makedirs(OUT_DIR, exist_ok=True)

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    assert p.wait() == 0, 'command failed: ' + cmd[:80]

sh('nvidia-smi --query-gpu=name,memory.total --format=csv; echo "CPUs: $(nproc)"; free -g | head -2')
DATA = r'''mkdir -p /content/wnn && cd /content/wnn
check() {
python3 - <<'EOF'
import sys, pyarrow.parquet as pq
ok = True
for f, n in {'train': 10607, 'valid': 1873, 'valid_mask': 1873}.items():
    try:
        g = pq.ParquetFile(f'datasets/{f}.parquet').metadata.num_row_groups
        print(f'{f}: {g} sequences', 'OK' if g == n else f'(expected {n})'); ok &= g == n
    except Exception as e:
        print(f'{f}: BROKEN ({type(e).__name__}: {e})'); ok = False
sys.exit(0 if ok else 1)
EOF
}
if check; then echo 'data OK, nothing to do'; else
  rm -rf datasets
  URL=https://files.wundernn.io/wnn_connectome_starterpack.tar.gz
  for i in $(seq 10); do   # -C - resumes where a dropped connection stopped
    curl -L -C - --retry 20 --retry-all-errors -sS -o pack.tar.gz "$URL" && break
    echo "download interrupted, resuming ($i)"; sleep 5
  done
  ls -la pack.tar.gz
  tar -xzf pack.tar.gz --strip-components=1 --warning=no-unknown-keyword && find . -name '._*' -delete
  if check; then rm pack.tar.gz; echo 'data OK'; else echo 'DATA STILL BROKEN: delete /content/wnn/pack.tar.gz and re-run this cell'; exit 1; fi
fi
du -sh datasets'''
sh(DATA)
sh('pip -q install onnx onnxruntime')
os.chdir('/content/wnn')
# sweep 6's notebook carries the pipeline code; only trainer.py changed since (per-model mini-batches)
import json, os, glob
src = None
for cand in ['/content/drive/MyDrive/wunder_challenge/connectome_sweep6.ipynb'] + glob.glob('/content/drive/MyDrive/*/wunder_challenge/connectome_sweep6.ipynb'):
    if os.path.exists(cand): src = cand; break
assert src, 'connectome_sweep6.ipynb not found in Drive'
os.makedirs('/content/wnn/pipeline', exist_ok=True); os.makedirs('/content/wnn/analysis', exist_ok=True)
n = 0
for c in json.load(open(src))['cells']:
    s = ''.join(c['source'])
    if c['cell_type'] == 'code' and s.startswith('%%writefile /content/wnn/'):
        path, _, body = s.partition('\n')
        path = path.split(' ', 1)[1].strip()
        open(path, 'w').write(body if body.endswith('\n') or not body else body + '\n'); n += 1
print(n, 'files written from', src)
PATCH = [('    feats: bool = False         # also feed the 36 extra causal features (pipeline/features.py)\n', '    feats: bool = False         # also feed the 36 extra causal features (pipeline/features.py)\n    micro_batch: int | None = None  # sequences per optimizer step for this model (default: the whole data batch)\n'), ('        total_steps = self.epochs * n_batches * math.ceil(20000 / self.chunk)', '        self.micro = cfg.micro_batch or common.batch\n        total_steps = (self.epochs * n_batches * math.ceil(common.batch / self.micro)  # upper bound on steps\n                       * math.ceil(20000 / self.chunk))'), ('                h = xp = None\n                losses = []\n                for s in range(0, X.shape[1], e.chunk):\n                    x, y = Xe[:, s:s + e.chunk], Ye[:, s:s + e.chunk]\n                    p, h = e.net(x, h, xp)\n                    lo = max(0, WARMUP - s)  # skip warm-up rows\n                    if lo < x.shape[1]:\n                        pf, yf = p[:, lo:].reshape(-1, 2), y[:, lo:].reshape(-1, 2)\n                        rw = We[:, s:s + e.chunk][:, lo:].reshape(-1) if We is not None else None\n                        lam = e.cfg.distill\n                        loss = compute_loss(e.cfg.loss, pf, yf, e.cfg.alpha, rw) if lam < 1 else 0.0\n                        if lam > 0:\n                            tf = Te[:, s:s + e.chunk][:, lo:].reshape(-1, 2)\n                            loss = (1 - lam) * loss + lam * distill_loss(pf, tf, yf, e.cfg.distill_weight, rw)\n                        e.opt.zero_grad(set_to_none=True)\n                        loss.backward()\n                        torch.nn.utils.clip_grad_norm_(e.net.parameters(), common.clip_grad)\n                        e.opt.step()\n                        e.sched.step()\n                        e.update_ema()\n                        losses.append(loss.item())\n                    h = detach_state(h)\n                    xp = x[:, -1]\n                e.log(event="batch", epoch=epoch, batch=bi, loss=round(float(np.mean(losses)), 5),\n                      lr=round(e.sched.get_last_lr()[0], 6))', '                losses = []\n                for m0 in range(0, len(Xe), e.micro):  # mini-batches of whole sequences, each its own TBPTT pass\n                    Xm, Ym = Xe[m0:m0 + e.micro], Ye[m0:m0 + e.micro]\n                    Tm = Te[m0:m0 + e.micro] if Te is not None else None\n                    Wm = We[m0:m0 + e.micro] if We is not None else None\n                    h = xp = None\n                    for s in range(0, X.shape[1], e.chunk):\n                        x, y = Xm[:, s:s + e.chunk], Ym[:, s:s + e.chunk]\n                        p, h = e.net(x, h, xp)\n                        lo = max(0, WARMUP - s)  # skip warm-up rows\n                        if lo < x.shape[1]:\n                            pf, yf = p[:, lo:].reshape(-1, 2), y[:, lo:].reshape(-1, 2)\n                            rw = Wm[:, s:s + e.chunk][:, lo:].reshape(-1) if Wm is not None else None\n                            lam = e.cfg.distill\n                            loss = compute_loss(e.cfg.loss, pf, yf, e.cfg.alpha, rw) if lam < 1 else 0.0\n                            if lam > 0:\n                                tf = Tm[:, s:s + e.chunk][:, lo:].reshape(-1, 2)\n                                loss = (1 - lam) * loss + lam * distill_loss(pf, tf, yf, e.cfg.distill_weight, rw)\n                            e.opt.zero_grad(set_to_none=True)\n                            loss.backward()\n                            torch.nn.utils.clip_grad_norm_(e.net.parameters(), common.clip_grad)\n                            e.opt.step()\n                            e.sched.step()\n                            e.update_ema()\n                            losses.append(loss.detach())  # no GPU sync per step\n                        h = detach_state(h)\n                        xp = x[:, -1]\n                e.log(event="batch", epoch=epoch, batch=bi, loss=round(float(torch.stack(losses).mean()), 5),\n                      lr=round(e.sched.get_last_lr()[0], 6), steps=len(losses))')]
t = open('/content/wnn/pipeline/trainer.py').read()
if 'micro_batch' not in t:
    for a, b in PATCH:
        assert t.count(a) == 1, 'trainer.py patch does not apply: ' + a[:60]
        t = t.replace(a, b)
    open('/content/wnn/pipeline/trainer.py', 'w').write(t)
import importlib, sys; sys.path.insert(0, '/content/wnn')
from pipeline.trainer import ExpConfig
assert 'micro_batch' in ExpConfig.__dataclass_fields__
print('trainer.py has per-model mini-batches')
sh('cd /content/wnn && python -m pipeline.fast_eval --baseline --valid-seqs 300')
MASK, CACHE = f'{OUT_DIR}/mask_model', f'{OUT_DIR}/cache'
ROWW, SPLIT = '/content/wnn/mask_train.npy', f'{OUT_DIR}/mask_model/split.json'
sh(f'cp {CACHE}/mask_train.npy {ROWW} && ls -la {ROWW}')
import json
experiments = [{'name': 'b256_c500', 'mask_weight': 0.5},
 {'name': 'b64_c500', 'mask_weight': 0.5, 'micro_batch': 64},
 {'name': 'b32_c500', 'mask_weight': 0.5, 'micro_batch': 32},
 {'name': 'b256_c2000', 'mask_weight': 0.5, 'chunk': 2000},
 {'name': 'b64_c2000', 'mask_weight': 0.5, 'micro_batch': 64, 'chunk': 2000},
 {'name': 'b32_c2000', 'mask_weight': 0.5, 'micro_batch': 32, 'chunk': 2000},
 {'name': 'b32_c5000', 'mask_weight': 0.5, 'micro_batch': 32, 'chunk': 5000},
 {'name': 'b32_c20000', 'mask_weight': 0.5, 'micro_batch': 32, 'chunk': 20000},
 {'name': 'b32_c500_lr1e-3', 'mask_weight': 0.5, 'micro_batch': 32, 'lr': 0.001}]
json.dump(experiments, open('experiments.json', 'w'), indent=1)
print(len(experiments), 'experiments')
SWEEP = f'{OUT_DIR}/sweep7'
sh(f"""set -o pipefail; cd /content/wnn && python -m pipeline.multi_train --config experiments.json --epochs 4 --evals-per-epoch 1 --threads 8 --cache-eval --row-weights {ROWW} --valid-split {SPLIT} --out-dir {SWEEP} 2>&1 | grep -v '"event": "batch"'""")
import pandas as pd
rows = [json.loads(l) for l in open(f'{SWEEP}/summary.jsonl')]
df = pd.DataFrame(rows).drop_duplicates('name', keep='last')
print(df[['name', 'valid_B_scored', 'valid_B_all', 'valid_scored', 'holdout_all']].sort_values('valid_B_scored', ascending=False).to_string(index=False))


Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
CPUs: 12
               total        used        free      shared  buff/cache   available
Mem:              83           1          78           0           3          81
train: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/train.parquet'. Detail: [errno 2] No such file or directory)
valid: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/valid.parquet'. Detail: [errno 2] No such file or directory)
valid_mask: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/valid_mask.parquet'. Detail: [errno 2] No such file or directory)
-rw-r--r-- 1 root root 33737976954 Oct 28 08:48 pack.tar.gz
train: 10607 sequences OK
valid: 1873 sequences OK
valid_mask: 1873 sequences OK
data OK
33G	datasets
16 files written from /content/drive/MyDrive/wunder_challenge/connectome_sweep6.ipynb
trainer.py has per-model mini-batches
{
 "valid": {
  "scored": 